In [ ]:
import sys, os, time
sys.path.append(os.getcwd())

import pandas as pd
import numpy as np

from decision_tree import DecisionTree
from random_forest import RandomForest
from metric_tracker import MetricTracker
from model_visualizer import ModelVisualizer

from itertools import product
from typing import Dict, List, Any

from sklearn.model_selection import (
    KFold,
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)
from sklearn.metrics import make_scorer, f1_score

In [ ]:
dataset_path = "../dataset/diabetes_binary_5050split_health_indicators_BRFSS2023.csv"
dataset = pd.read_csv(dataset_path)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)
dataset.head()

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
tracker = MetricTracker()
viz = ModelVisualizer()

models = {
    "rf_probs": RandomForest(store_probs=True),
    "rf_labels": RandomForest(store_probs=False),
    "dt_probs": DecisionTree(store_probs=True),
    "dt_labels": DecisionTree(store_probs=False)
}
metrics = {}

def cross_validate(model_name, model, X, y, kf):
    tracker.clear_cv_history()
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train)

        row = tracker.compute_cv_metrics(model_name, model, X_val, y_val)
        row["fold"] = fold_idx
        tracker.add_cv_row(row)
    return tracker.cv_to_df()

for model_name, model in models.items():
    print(f"Cross validating {model_name}...")
    start_time = time.time()

    metrics[f"cv_{model_name}"] = cross_validate(model_name, model, X_train, y_train, kf)
    
    runtime = time.time() - start_time
    print(f"Cross validation finished in {runtime}\n") 

In [ ]:
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'store_probs': [True]
}

# Create the GridSearchCV object
rf_grid = GridSearchCV(
    estimator=RandomForest(),
    param_grid=rf_param_grid,
    cv=5,                          # 5-fold cross-validation
    scoring='f1',                  # Optimize for F1 score
    n_jobs=-1,                     # Use all CPU cores (PARALLEL!)
    verbose=2,                     # Show progress
    return_train_score=True,
    refit=True                     # Refit best model on full training set
)

# Fit (this does all the cross-validation automatically)
print("\nStarting Grid Search...")
start_time = time.time()
rf_grid.fit(X_train, y_train)
elapsed = time.time() - start_time

print(f"\nGrid Search completed in {elapsed:.2f} seconds")
print(f"Best F1 Score: {rf_grid.best_score_:.4f}")
print(f"Best Parameters: {rf_grid.best_params_}")

# Access all results as DataFrame
results_df = pd.DataFrame(rf_grid.cv_results_)
print("\nTop 5 configurations:")
print(results_df[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head())

# The best model is already fitted and available
best_rf = rf_grid.best_estimator_

# Evaluate on test set
tracker = MetricTracker()
test_metrics = tracker.compute_test_metrics("best_rf", best_rf, X_test, y_test)
print("\nTest Set Performance:")
print(pd.DataFrame([test_metrics]))

In [ ]:
rf_random = RandomizedSearchCV(
    estimator=RandomForest(),
    param_distributions=rf_param_grid,
    n_iter=20,                     # Try 20 random combinations
    cv=5,
    scoring='f1',
    n_jobs=-1,                     # PARALLEL
    verbose=2,
    random_state=42,
    return_train_score=True,
    refit=True
)

print("\nStarting Random Search...")
start_time = time.time()
rf_random.fit(X_train, y_train)
elapsed = time.time() - start_time

print(f"\nRandom Search completed in {elapsed:.2f} seconds")
print(f"Best F1 Score: {rf_random.best_score_:.4f}")
print(f"Best Parameters: {rf_random.best_params_}")

In [ ]:
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

rf_param_grid_small = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt'],
    'store_probs': [True]
}

rf_multi = GridSearchCV(
    estimator=RandomForest(),
    param_grid=rf_param_grid_small,
    cv=3,
    scoring=scoring,
    refit='f1',                    # Refit using F1 as the criterion
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("\nStarting Multi-Metric Grid Search...")
rf_multi.fit(X_train, y_train)

print(f"\nBest F1 Score: {rf_multi.best_score_:.4f}")
print(f"Best Parameters: {rf_multi.best_params_}")

# View all metrics for best model
results_multi = pd.DataFrame(rf_multi.cv_results_)
best_idx = rf_multi.best_index_

print("\nAll metrics for best configuration:")
for metric in scoring.keys():
    mean_score = results_multi.loc[best_idx, f'mean_test_{metric}']
    std_score = results_multi.loc[best_idx, f'std_test_{metric}']
    print(f"  {metric}: {mean_score:.4f} (+/- {std_score:.4f})")

In [ ]:
dt_param_grid = {
    'max_depth': [3, 5, 7, 10, 15, 20],
    'min_samples_split': [2, 5, 10, 20, 50],
    'store_probs': [True, False]
}

dt_grid = GridSearchCV(
    estimator=DecisionTree(),
    param_grid=dt_param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    refit=True
)

print("\nStarting Decision Tree Grid Search...")
dt_grid.fit(X_train, y_train)

print(f"\nBest F1 Score: {dt_grid.best_score_:.4f}")
print(f"Best Parameters: {dt_grid.best_params_}")

best_dt = dt_grid.best_estimator_
test_metrics_dt = tracker.compute_test_metrics("best_dt", best_dt, X_test, y_test)
print("\nTest Set Performance:")
print(pd.DataFrame([test_metrics_dt]))

In [ ]:
custom_f1_scorer = make_scorer(f1_score, average='macro')

rf_param_grid_small = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt'],
    'store_probs': [True]
}

rf_custom = GridSearchCV(
    estimator=RandomForest(),
    param_grid=rf_param_grid_small,
    cv=3,
    scoring=custom_f1_scorer,      # Use custom scorer
    n_jobs=-1,
    verbose=1,
    refit=True
)

print("\nStarting Grid Search with custom F1 (macro average)...")
rf_custom.fit(X_train, y_train)

print(f"\nBest Macro F1 Score: {rf_custom.best_score_:.4f}")
print(f"Best Parameters: {rf_custom.best_params_}")

In [ ]:
rf_single = RandomForest(n_estimators=100, max_depth=10, store_probs=True)

scores = cross_val_score(
    rf_single,
    X_train, y_train,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

print(f"5-Fold CV F1 Scores: {scores}")
print(f"Mean F1: {scores.mean():.4f} (+/- {scores.std():.4f})")

In [ ]:
import matplotlib.pyplot as plt

# Plot GridSearchCV results
results = pd.DataFrame(rf_grid.cv_results_)

# Plot relationship between max_depth and score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Max Depth vs Score
for n_est in [50, 100, 200]:
    mask = results['param_n_estimators'] == n_est
    subset = results[mask].groupby('param_max_depth')['mean_test_score'].mean()
    axes[0].plot(subset.index, subset.values, marker='o', label=f'n_est={n_est}')

axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('Mean F1 Score')
axes[0].set_title('F1 Score vs Max Depth')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Top 10 configurations
top_10 = results.nsmallest(10, 'rank_test_score')
x_pos = np.arange(len(top_10))
axes[1].bar(x_pos, top_10['mean_test_score'], yerr=top_10['std_test_score'], 
            capsize=5, alpha=0.7, color='steelblue')
axes[1].set_xlabel('Configuration Rank')
axes[1].set_ylabel('Mean F1 Score')
axes[1].set_title('Top 10 Configurations')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(range(1, len(top_10)+1))
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()